# QZAP Text Analysis: Named Entity Recognition and Mapping

## About This Notebook
This notebook extends the analysis from `02_data_pipeline.ipynb` by extracting
place names from each zine using Named Entity Recognition (NER) and visualizing
them on an interactive map.

## Data Access
This notebook loads the CSV file saved by `02_data_pipeline.ipynb`, located at
`output/extracted_text/zines_data.csv`. Run `02_data_pipeline.ipynb` first to
generate that file.

## Research Question
Where are these zines located, which places are named, and what geographies of
queer life do they reflect or construct? Cities, neighborhoods, and regions carry histories of visibility, danger, community, and displacement. This notebook offers a computational starting point for asking
whether the archive skews toward particular geographies.

In [1]:
# Import libraries
import spacy
import folium
import pandas as pd
from geopy.geocoders import Nominatim
import time

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

print("Libraries loaded!")

Libraries loaded!


In [2]:
# Load the zine data I saved from the pipeline notebook
zines_df = pd.read_csv("../output/extracted_text/zines_data.csv")
print(zines_df[["title", "year", "word_count"]])

                                               title  year  word_count
0                                    behind the bars  2006        2224
1                                           doris 21  2003        2680
2                                  empty orchestra 1  2012        2510
3                                            gaysi 1  2011        4404
4                                     gender trash 1  1993        5147
5                                       going homo 3  1994        3996
6                            queer zine explosion 20  2003        2474
7                          the anarchistic queer 1.1  2011        1111
8                          timtum - a trans jew zine  1999        6816
9  we have always existed transgender people thro...  2022         667


## Named Entity Recognition: Place Extraction
I use spaCy's NER model to extract place names from each zine's text.
spaCy labels geopolitical entities as `GPE` (countries, cities, states) and
locations as `LOC` (rivers, mountains, regions). I extract both.

Note: spaCy's NER model was trained on newspaper text, not zine text. OCR noise
in my data means some false positives are expected and must be filtered manually.

In [3]:
# Extract place names from each zine using spaCy NER
def extract_places(text):
    doc = nlp(text)
    places = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]
    return places

zines_df["places"] = zines_df["full_text"].apply(extract_places)

# Preview results
for index, row in zines_df.iterrows():
    print("=== ", row["title"], " ===")
    print(row["places"])
    print()

===  behind the bars  ===
['Southeast Portland', 'Hawaii', 'constanL', 'Hawaii', 'procedur·Ps', 'gf"nitalia', 'Cdrmot', 'hwnan', 'llwnan', 'California', 'U.S.', 'California', 'Hawaii', 'Fort', 'Kansas', 'Oregon', 'Belmont', 'Portland']

===  doris 21  ===
['Guatemala', 'stu!f', "T'l1", 'NC', 'Gertie', 'cou.ld', 'wo••n', '•••', 'na.me', '•••', 'haTe', 'y-\\OvJ', 'Guatemala', 'Russia', 'Siberia', 'Cesar Chsvez', 'Berto', 'US', 'U.S.', "I'\\", 'the1', 'USA', 'u.s.', 'u.s.', 'Guatamala', 'U.S.', 'Guatemala', 'Berto', 'Guatemala City', 'The Island of the Blue\nDolphins', 'Cosmopolitan', 'U.S.A.', 'md', 'Erotic', '~f~~-', 'worl~', 'haTe', 'haTe', "WA'tCA"]

===  empty orchestra 1  ===
['US', 'Opera', 'proce88', '•••', 'Pabst', 'Southland', 'the Big City of Lexington', 'Eleventh', 'Eastland', 'Eastland', 'Posaum Kingdom', 'South Atrica', 'China', 'Lexington', "St. Patrick's Day", 'Sometimea', 'Louisville', 'Brandon', 'Stephen']

===  gaysi 1  ===
['America', 'North America', 'South India', 'D

## Cleaning NER Output
The raw NER output contains OCR artifacts incorrectly identified as place names
(e.g., `constanL`, `hwnan`, `proce88`). I apply an alphabetical filter to remove
obvious noise, then manually verify the remaining results against the original zine text.

In [4]:
# Filter out OCR noise — keep only alphabetical tokens longer than 2 characters
def clean_places(places):
    cleaned = []
    for place in places:
        if place.replace(" ", "").replace(".", "").replace("'", "").isalpha():
            if len(place) > 2:
                cleaned.append(place)
    return cleaned

zines_df["clean_places"] = zines_df["places"].apply(clean_places)

for index, row in zines_df.iterrows():
    print("===", row["title"], "===")
    print(row["clean_places"])
    print()

=== behind the bars ===
['Southeast Portland', 'Hawaii', 'constanL', 'Hawaii', 'Cdrmot', 'hwnan', 'llwnan', 'California', 'U.S.', 'California', 'Hawaii', 'Fort', 'Kansas', 'Oregon', 'Belmont', 'Portland']

=== doris 21 ===
['Guatemala', 'Gertie', 'cou.ld', 'na.me', 'haTe', 'Guatemala', 'Russia', 'Siberia', 'Cesar Chsvez', 'Berto', 'U.S.', 'USA', 'u.s.', 'u.s.', 'Guatamala', 'U.S.', 'Guatemala', 'Berto', 'Guatemala City', 'Cosmopolitan', 'U.S.A.', 'Erotic', 'haTe', 'haTe', "WA'tCA"]

=== empty orchestra 1 ===
['Opera', 'Pabst', 'Southland', 'the Big City of Lexington', 'Eleventh', 'Eastland', 'Eastland', 'Posaum Kingdom', 'South Atrica', 'China', 'Lexington', "St. Patrick's Day", 'Sometimea', 'Louisville', 'Brandon', 'Stephen']

=== gaysi 1 ===
['America', 'North America', 'South India', 'Delhi', 'India', 'Priya', 'holdhlJJ', 'Ashrarnmn', 'stmt', 'India', 'Hindi', 'India', 'mille', 'hai', 'hai', 'Bolivia', 'Shani', 'brinjaf', 'India', 'California', 'Illinois']

=== gender trash 1 ===
['

In [5]:
# Automated filtering still returns false positives (e.g. "Opera", "Brandon")
# Manually verify real places from NER output
verified_places = {
    "behind the bars":        ["Southeast Portland", "Hawaii", "California", "Kansas", "Oregon", "Portland", "Belmont"],
    "doris 21":               ["Guatemala", "Russia", "Siberia", "Guatemala City", "United States"],
    "empty orchestra 1":      ["Lexington", "Louisville", "South Africa", "China"],
    "gaysi 1":                ["America", "South India", "Delhi", "India", "Bolivia", "California", "Illinois"],
    "gender trash 1":         ["East York", "Toronto", "Hollywood", "Ireland", "Florida", "Montreal", "New York City", "New York"],
    "going homo 3":           ["Tucson", "Arizona", "Colorado", "America", "Germany", "Cleveland", "Ohio", "Kentucky", "Chicago", "Chapel Hill", "New Zealand", "Bethlehem"],
    "queer zine explosion 20":["Arlington", "Richmond", "Italy", "Vancouver", "New York", "Middletown", "Portland", "Tucson", "Switzerland", "Toronto", "Cambridge", "Minneapolis", "Turkey", "Europe", "Quebec", "Kingston", "Galveston"],
    "the anarchistic queer 1.1": [],
    "timtum - a trans jew zine": ["Europe", "Poland", "France", "Spain", "Detroit", "Paris", "New York", "Egypt", "Israel", "Eastern Europe", "North Africa", "Morocco", "Turkey", "Greece", "United States"],
    "we have always existed transgender people through history": ["Hudson River", "Malaysia", "India", "Texas", "United States"]
}

zines_df["verified_places"] = zines_df["title"].map(verified_places)

for index, row in zines_df.iterrows():
    print("===", row["title"], "===")
    print(row["verified_places"])
    print()

=== behind the bars ===
['Southeast Portland', 'Hawaii', 'California', 'Kansas', 'Oregon', 'Portland', 'Belmont']

=== doris 21 ===
['Guatemala', 'Russia', 'Siberia', 'Guatemala City', 'United States']

=== empty orchestra 1 ===
['Lexington', 'Louisville', 'South Africa', 'China']

=== gaysi 1 ===
['America', 'South India', 'Delhi', 'India', 'Bolivia', 'California', 'Illinois']

=== gender trash 1 ===
['East York', 'Toronto', 'Hollywood', 'Ireland', 'Florida', 'Montreal', 'New York City', 'New York']

=== going homo 3 ===
['Tucson', 'Arizona', 'Colorado', 'America', 'Germany', 'Cleveland', 'Ohio', 'Kentucky', 'Chicago', 'Chapel Hill', 'New Zealand', 'Bethlehem']

=== queer zine explosion 20 ===
['Arlington', 'Richmond', 'Italy', 'Vancouver', 'New York', 'Middletown', 'Portland', 'Tucson', 'Switzerland', 'Toronto', 'Cambridge', 'Minneapolis', 'Turkey', 'Europe', 'Quebec', 'Kingston', 'Galveston']

=== the anarchistic queer 1.1 ===
[]

=== timtum - a trans jew zine ===
['Europe', 'Poland

## Geocoding and Mapping
I convert each verified place name into geographic coordinates using `geopy`,
then visualize the results on an interactive map using `folium`. Each zine is
assigned a color so geographic patterns can be compared across zines.

In [6]:
# Geocode all verified places
geolocator = Nominatim(user_agent="qzap_mapping")
map_data = []

for index, row in zines_df.iterrows():
    for place in row["verified_places"]:
        location = geolocator.geocode(place)
        time.sleep(1)
        if location:
            map_data.append({
                "title": row["title"],
                "place": place,
                "lat": location.latitude,
                "lon": location.longitude
            })
            print(f"Found: {place} -> {location.latitude}, {location.longitude}")
        else:
            print(f"Not found: {place}")

map_df = pd.DataFrame(map_data)
map_df = map_df.merge(zines_df[["title", "year"]], on="title", how="left")  # add year column
map_df

Found: Southeast Portland -> 45.5112811, -122.6653763
Found: Hawaii -> 19.5938015, -155.4283701
Found: California -> 36.7014631, -118.755997
Found: Kansas -> 38.27312, -98.5821872
Found: Oregon -> 43.9792797, -120.737257
Found: Portland -> 45.5202471, -122.674194
Found: Belmont -> 48.4095949, 7.234863
Found: Guatemala -> 15.5855545, -90.345759
Found: Russia -> 64.6863136, 97.7453061
Found: Siberia -> 45.1685441, 8.0902162
Found: Guatemala City -> 14.6416142, -90.5132836
Found: United States -> 39.7837304, -100.445882
Found: Lexington -> 38.0464066, -84.4970393
Found: Louisville -> 38.2542376, -85.759407
Found: South Africa -> -28.8166236, 24.991639
Found: China -> 34.5412252, 108.9237067
Found: America -> 39.7837304, -100.445882
Found: South India -> 17.3615965, 78.4276746
Found: Delhi -> 28.6138954, 77.2090057
Found: India -> 22.3511148, 78.6677428
Found: Bolivia -> -17.0568696, -64.9912286
Found: California -> 36.7014631, -118.755997
Found: Illinois -> 40.0796606, -89.4337288
Found: 

,title,place,lat,lon,year
0,behind the bars,Southeast Portland,45.511281,-122.665376,2006
1,behind the bars,Hawaii,19.593802,-155.428370,2006
2,behind the bars,California,36.701463,-118.755997,2006
3,behind the bars,Kansas,38.273120,-98.582187,2006
4,behind the bars,Oregon,43.979280,-120.737257,2006
...,...,...,...,...,...
75,we have always existed transgender people thro...,Hudson River,42.611240,-73.760755,2022
76,we have always existed transgender people thro...,Malaysia,4.569375,102.265682,2022
77,we have always existed transgender people thro...,India,22.351115,78.667743,2022
78,we have always existed transgender people thro...,Texas,31.263890,-98.545612,2022


In [8]:
# Build styled interactive map color coded by zine
colors = {
    "behind the bars":        "red",
    "doris 21":               "orange",
    "empty orchestra 1":      "gold",
    "gaysi 1":                "green",
    "gender trash 1":         "lightblue",
    "going homo 3":           "blue",
    "queer zine explosion 20":"purple",
    "the anarchistic queer 1.1": "pink",
    "timtum - a trans jew zine": "gray",
    "we have always existed transgender people through history": "brown"
}

qzap_map = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="CartoDB positron"
)

for index, row in map_df.iterrows():
    color = colors.get(row["title"], "black")
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=10,
        color="white",
        weight=2,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{row['place']}</b><br><i>{row['title']}</i> ({row['year']})",
            max_width=200
        )
    ).add_to(qzap_map)

# Add legend
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background-color: white; padding: 15px; border-radius: 8px;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.3); font-family: Arial; font-size: 13px;">
     <b style="font-size:14px;">QZAP Zines</b><br><br>
     <span style="color:red;">&#9679;</span> behind the bars<br>
     <span style="color:orange;">&#9679;</span> doris 21<br>
     <span style="color:gold;">&#9679;</span> empty orchestra 1<br>
     <span style="color:green;">&#9679;</span> gaysi 1<br>
     <span style="color:lightblue;">&#9679;</span> gender trash 1<br>
     <span style="color:blue;">&#9679;</span> going homo 3<br>
     <span style="color:purple;">&#9679;</span> queer zine explosion 20<br>
     <span style="color:pink;">&#9679;</span> the anarchistic queer 1.1<br>
     <span style="color:gray;">&#9679;</span> timtum - a trans jew zine<br>
     <span style="color:brown;">&#9679;</span> we have always existed transgender people through history
</div>
"""
qzap_map.get_root().html.add_child(folium.Element(legend_html))

qzap_map.save("../output/qzap_map.html")
print("Map saved to output/qzap_map.html")
qzap_map

Map saved to output/qzap_map.html


## Results

The map reveals a strong geographic concentration in North America and Europe.
Of the 80 geocoded places across all ten zines, the vast majority fall in the
United States, Canada, and Western Europe. This is itself a finding: the
geographies of queer life that appear in this sample of QZAP zines are not
evenly distributed across the world.

Several patterns stand out:

**North American dominance.** Almost every zine references at least one US
location. Behind the Bars is the most locally anchored, with places clustered
tightly around Portland and Oregon. Going Homo is similarly rooted in one city:
Tucson appears more than any other place in the entire sample. Queer Zine
Explosion, as a review compilation, produces the widest US footprint, with
cities scattered across the country reflecting the geographic spread of the
zines it covers.

**International references are the exception, not the rule.** When zines do
reach beyond North America, it is almost always tied to a specific political
or cultural project: Gaysi references India because it centers South Asian
queer identity, TimTum references the Middle East and Eastern Europe through
the lens of Jewish diaspora, and We Have Always Existed names Malaysia and
India as part of recovering global transgender history. These are purposeful
references, not incidental ones.

**Place references tend to be local and specific.** Rather than referencing
the world broadly, most zines name the particular places they are from or
writing about. Behind the Bars clusters around Portland and Oregon. Going Homo
is anchored almost entirely in Tucson. Gender Trash maps Toronto street by
street. This pattern suggests that zines are deeply place-based objects and
their geographic imagination is rooted in the communities that made them,
not in an abstract or global sense of queer space.

**What this answers and what it doesn't.** The map offers a partial answer
to the research question. The places named in these zines do skew heavily
toward North America and Western Europe, suggesting that the geographic
imagination of this sample reflects the archive's own demographic base. But
the map only captures places that survived text extraction and manual
verification. Zines are also deeply visual objects: hand-drawn maps,
photographs of neighborhoods, and illustrated geographies are entirely
invisible to this method. The computational picture is real but incomplete.

## Connecting Geography and Vocabulary

Reading the map alongside the TF-IDF results from `02_data_pipeline.ipynb`
reveals a pattern that neither method surfaces on its own: zines with a
narrower geographic footprint tend to also have more locally-specific
vocabulary, while zines with a wider geographic reach tend to have more
ideological or identity-based vocabulary.

The clearest examples are at opposite ends of the spectrum. Behind the Bars
and Going Homo are the most geographically concentrated zines in the sample:
Behind the Bars clusters almost entirely around Portland and Oregon, and Going
Homo names Tucson more than any other place in the entire dataset. Their
TF-IDF results match this local anchoring: their most distinctive terms are
tied to specific activist contexts and subcultural scenes rather than broad
political frameworks.

At the other end, the zines with the widest geographic reach, like Gaysi, TimTum,
and We Have Always Existed, also produce the most expansive and
identity-spanning vocabulary in the TF-IDF results. Gaysi's top terms move
between South Asian cultural vocabulary and LGBTQ discourse. TimTum's span
Jewish diaspora geography in both its place references and its terminology.
We Have Always Existed names places across Southeast Asia, South Asia, and
the US, and its vocabulary reflects that global scope through its focus on
historical figures across cultures.

This connection suggests something about what zines do geographically: a zine
rooted in one city tends to speak the language of that city's specific
community, while a zine with a diasporic or global politics reaches for
vocabulary that can travel across places. TF-IDF and NER were applied separately, but their results point in the
same direction.

## Reflection

**What worked:**
- spaCy's NER successfully identified real place names across all ten zines,
  though with varying reliability depending on OCR quality
- Geocoding with `geopy` located all 80 verified places accurately with no
  failed lookups
- The color-coded map makes it easy to compare geographic spread across zines
  at a glance. Some zines produce tight local clusters while others scatter
  points across continents

**What didn't work:**
- NER produced many false positives due to OCR noise, requiring manual
  verification of every result. This was the most time-consuming part of the
  notebook
- The automated alphabetical filter helped but was not sufficient on its own.
  Names like `Opera`, `Brandon`, and `Sometimea` still passed through and had
  to be removed by hand
- Several zines with heavy OCR noise produced sparse or unreliable place lists.
  The Anarchistic Queer returned only one token (`fonn`) and could not be mapped
  at all. Its absence from the map reflects a limitation of the method, not the
  zine's actual content
- Some real place names were too garbled to geocode reliably and had to be
  excluded. `Les Aftceles` (Los Angeles) and `San l'r.tndseo` (San Francisco)
  are two examples where the OCR error was clear but the corrupted string could
  not be passed to the geocoder

**What I learned:**
- NER is a machine learning method trained on newspaper text and does not
  transfer cleanly to zine text, especially when OCR quality is poor. The
  gap between what NER can do and what zines actually are is itself a
  meaningful finding about the limits of computational methods in this domain
- The curatorial decisions made during manual verification, such as choosing which
  places count as real and which are noise, are interpretive acts, not
  neutral ones. A different researcher making different calls would produce
  a different map
- A zine's visual content (photographs, hand-drawn maps, illustrated
  geographies) is entirely invisible to NER, meaning computational place
  extraction captures only a fraction of each zine's actual geographic
  imagination
- Geographic skew is itself a finding. The concentration of US references
  across the sample reflects both the archive's North American base and the
  limits of text extraction on image-heavy zines whose places never made it
  into extractable text